In [16]:
!pip3 install torch transformers accelerate sentencepiece pandas tqdm

  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 MB 11.9 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 11.6 MB/s  0:00:01m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 9.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 11.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 11.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.6 MB/s  0:00:00
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 12.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.

In [17]:
import torch
import pandas as pd
from tqdm import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    pipeline
)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [26]:
############################################
# CONFIG
############################################
INPUT_CSV = "test.csv"
OUTPUT_CSV = "test_translated_scored.csv"
BATCH_SIZE = 16
MAX_INPUT_LEN = 512
MAX_NEW_TOKENS = 256

TRANSLATION_MODEL = "facebook/nllb-200-distilled-600M"
SENTIMENT_MODEL = "nlptown/bert-base-multilingual-uncased-sentiment"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(DEVICE)

cpu


In [19]:
############################################
# LOAD TRANSLATION MODEL
############################################
print("Loading facebook/nllb-200-distilled-600M...")

mt_tokenizer = AutoTokenizer.from_pretrained(
    TRANSLATION_MODEL,
    use_fast=True
)

mt_model = AutoModelForSeq2SeqLM.from_pretrained(
    TRANSLATION_MODEL,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)

mt_model.eval()

Loading facebook/nllb-200-distilled-600M...


`torch_dtype` is deprecated! Use `dtype` instead!


M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-11): 12 x M2M100EncoderLayer(
          (self_attn): M2M100Attention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
       

In [20]:
############################################
# LOAD SENTIMENT MODEL
############################################
print("Loading sentiment model...")

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=SENTIMENT_MODEL,
    tokenizer=SENTIMENT_MODEL,
    device=0 if DEVICE == "cuda" else -1
)

Loading sentiment model...


Device set to use cpu


In [21]:
LANG_CODE_MAP = {
    "en": "eng_Latn",
    "de": "deu_Latn",
    "fr": "fra_Latn",
    "es": "spa_Latn",
    "ja": "jpn_Jpan",
    "zh": "zho_Hans"
}

In [28]:
############################################
# TRANSLATION FUNCTION
############################################
def translate_to_english(texts, src_lang, batch_size=8, max_length=512):
    """
    Translate list of texts from src_lang → English using NLLB
    CPU / GPU safe (Mac-compatible)
    """

    if src_lang not in LANG_CODE_MAP:
        raise ValueError(f"Unsupported language: {src_lang}")

    src_lang_code = LANG_CODE_MAP[src_lang]
    tgt_lang_code = "eng_Latn"

    mt_tokenizer.src_lang = src_lang_code
    forced_bos_token_id = mt_tokenizer.convert_tokens_to_ids(tgt_lang_code)

    translations = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        inputs = mt_tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        )

        with torch.no_grad():
            outputs = mt_model.generate(
                **inputs,
                forced_bos_token_id=forced_bos_token_id,
                max_new_tokens=256,
                num_beams=4,
                do_sample=False
            )

        decoded = mt_tokenizer.batch_decode(
            outputs,
            skip_special_tokens=True
        )

        translations.extend(decoded)

    return translations

In [23]:
############################################
# SENTIMENT SCORING FUNCTION
############################################
def get_sentiment_scores(texts):
    """
    Returns integer sentiment scores from 1 to 5
    Safely handles long texts by truncating to BERT max length (512)
    """

    results = sentiment_pipeline(
        texts,
        batch_size=32,
        truncation=True,     
        max_length=512       
    )

    scores = []
    for r in results:
        # label format: "1 star", "2 stars", ...
        score = int(r["label"].split()[0])
        scores.append(score)
    
    return scores

In [24]:
############################################
# MAIN PIPELINE
############################################

def process_reviews(df):
    translated_reviews = []
    sentiment_scores = []

    total_batches = (len(df) + BATCH_SIZE - 1) // BATCH_SIZE

    print(f"Starting processing: {len(df)} reviews, {total_batches} batches")

    for batch_idx in tqdm(range(0, len(df), BATCH_SIZE), desc="Processing batches"):
        batch = df.iloc[batch_idx:batch_idx + BATCH_SIZE]

        texts = batch["review_body"].fillna("").tolist()
        langs = batch["language"].tolist()

        translations_map = {}

        # -------------------------
        # Translation (per language)
        # -------------------------
        for lang in sorted(set(langs)):
            indices = [i for i, l in enumerate(langs) if l == lang]
            lang_texts = [texts[i] for i in indices]

            print(f"[Batch {batch_idx}] Translating {len(lang_texts)} reviews (lang={lang})")

            if lang == "en":
                translated = lang_texts
            else:
                translated = translate_to_english(lang_texts, lang)

            for i, t in zip(indices, translated):
                translations_map[i] = t

            print(f"[Batch {batch_idx}] Translation done (lang={lang})")

        ordered_translations = [translations_map[i] for i in range(len(batch))]

        # -------------------------
        # Sentiment scoring
        # -------------------------
        print(f"[Batch {batch_idx}] Scoring sentiment for {len(ordered_translations)} reviews")

        scores = []
        for j in range(0, len(ordered_translations), 32):
            sub_batch = ordered_translations[j:j + 32]
            sub_scores = get_sentiment_scores(sub_batch)
            scores.extend(sub_scores)

        print(f"[Batch {batch_idx}] Sentiment scoring completed")

        translated_reviews.extend(ordered_translations)
        sentiment_scores.extend(scores)

    print("Processing completed successfully ✅")

    return translated_reviews, sentiment_scores

In [29]:
############################################
# RUN
############################################
if __name__ == "__main__":
    print("Reading CSV...")
    df = pd.read_csv(INPUT_CSV)

    print("Translating and scoring reviews...")
    translated_reviews, sentiment_scores = process_reviews(df)

    df["review_body_en"] = translated_reviews
    df["sentiment_score_1_to_5"] = sentiment_scores

    print(f"Saving output to {OUTPUT_CSV}")
    df.to_csv(OUTPUT_CSV, index=False)

    print("Done")

Reading CSV...
Translating and scoring reviews...
Starting processing: 30000 reviews, 1875 batches


Processing batches:   0%|          | 0/1875 [00:00<?, ?it/s]

[Batch 0] Translating 16 reviews (lang=de)
[Batch 0] Translation done (lang=de)
[Batch 0] Scoring sentiment for 16 reviews


Processing batches:   0%|          | 1/1875 [00:27<14:15:21, 27.39s/it]

[Batch 0] Sentiment scoring completed
[Batch 16] Translating 16 reviews (lang=de)
[Batch 16] Translation done (lang=de)
[Batch 16] Scoring sentiment for 16 reviews


...

Processing batches: 100%|█████████▉| 1874/1875 [14:14:14<00:24, 24.71s/it]

[Batch 29968] Sentiment scoring completed
[Batch 29984] Translating 16 reviews (lang=zh)
[Batch 29984] Translation done (lang=zh)
[Batch 29984] Scoring sentiment for 16 reviews


Processing batches: 100%|██████████| 1875/1875 [14:14:55<00:00, 27.36s/it]

[Batch 29984] Sentiment scoring completed
Processing completed successfully ✅
Saving output to test_translated_scored.csv


Done


In [3]:
import pandas as pd

df_1 = pd.read_csv("test_translated_scored.csv")
df_2 = pd.read_csv("hofstede_country_scores.csv")

# Language to country mapping (based on Amazon marketplaces)
lang_country_map = {
    'de': 'Germany',
    'en': 'USA',      # English from amazon.com (US primary)
    'es': 'Spain',
    'fr': 'France', 
    'ja': 'Japan',
    'zh': 'China'
}

# Add 'country' column to df_1 based on language
df_1['country'] = df_1['language'].map(lang_country_map)

# Update country name for USA
df_2['country'] = df_2['country'].replace('United states', 'USA')

# Merge df_1 with df_2 on 'country' column
df_merged = df_1.merge(df_2, on='country', how='left')

df_merged.head()

,Unnamed: 0,review_id,product_id,reviewer_id,stars,review_body,review_title,language,product_category,review_body_en,sentiment_score_1_to_5,country,pdi,idv,mas,uai,lto,ivr
0,0,de_0784695,product_de_0572654,reviewer_de_0645436,1,"Leider, leider nach einmal waschen ausgebliche...",Leider nicht zu empfehlen,de,home,"Unfortunately, it looks super pretty, but unfo...",2,Germany,35,79,66,65,57.0,40.0
1,1,de_0759207,product_de_0567331,reviewer_de_0183703,1,zunächst macht der Anker Halter einen soliden ...,Gummierung nach 6 Monaten kaputt,de,wireless,"First of all, the anchor holder makes a solid ...",2,Germany,35,79,66,65,57.0,40.0
2,2,de_0711785,product_de_0482105,reviewer_de_0182152,1,Siegel sowie Verpackung war beschädigt und war...,Flohmarkt ware,de,industrial_supplies,Seal as well as packaging was damaged and ware...,1,Germany,35,79,66,65,57.0,40.0
3,3,de_0964430,product_de_0616480,reviewer_de_0991563,1,Habe dieses Produkt NIE erhalten und das Geld ...,Katastrophe,de,industrial_supplies,I NEVER received this product and the money wa...,1,Germany,35,79,66,65,57.0,40.0
4,4,de_0474538,product_de_0228702,reviewer_de_0316188,1,Die Träger sind schnell abgerissen,Reißverschluss klemmt,de,luggage,The carriers quickly collapsed.,2,Germany,35,79,66,65,57.0,40.0


In [4]:
# Simple mapping for discrete 1-5 values (most efficient)
nps_mapping = {1: 'detractor', 2: 'detractor', 3: 'passive', 4: 'promoter', 5: 'promoter'}

# Apply mapping directly (fastest for discrete values)
df_merged['nps_category_stars'] = df_merged['stars'].map(nps_mapping)
df_merged['nps_category_sentiment'] = df_merged['sentiment_score_1_to_5'].map(nps_mapping)

df_merged.to_csv("nps_raw.csv", index=False)